---

# Acumula por Dia numa Grade os Flashes do GLM do GOES-16/19 dos Arquivos de 20s que foram Baixados da Amazon

---

- `OBJETIVO`:
> Este código acumula os flashes para uma região e dia específico numa grade de 8km X 8km e salva num arquivo NETCDF. As ocorrências de flashes também são salvas  num dataframe contendo o tempo (tempo do primeiro evento do flash), latitude e longitude do flash.



- `DADOS DE ENTRADA`:
> Dados do sensor GLM do satélite [GOES-16](https://noaa-goes16.s3.amazonaws.com/index.html#GLM-L2-LCFA/) ou [GOES-19](https://noaa-goes19.s3.amazonaws.com/index.html#GLM-L2-LCFA/) fornecido pela AMAZON. Exemplo de nome do arquivo: `OR_GLM-L2-LCFA_G16_s20201820000000_e20201820000200_c20201820000224.nc`


- `DADOS DE SAÍDA`:
> 1. Arquivo NETCDF de flashes numa grade de 8km X 8km. Exemplo: `flash_glm_goes_2020-06-30.nc`
> 2. Dataframe do tempo, latitude e longitude do flash. Exemplo: `flash_glm_goes_2020-06-30.csv`


- `OBSERVAÇÕES`:
   > Tempo de processamento de 1 dia de dados = `1h21min43s`

- `REALIZADO POR`:
> Enrique V. Mattos - 03/10/2025

- `ATUALIZADO POR`:
> Enrique V. Mattos - 27/04/2026
---


# **1° Passo:** Preparando ambiente

In [1]:
# importa bibliotecas
import os
import time
import glob
import xarray as xr
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# monta drive
from google.colab import drive
drive.mount('/content/drive')

# diretório raiz
dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/000_CODIGOS_REFERENCIA/03_RELAMPAGOS_SATELITE_REFERENCIA'

# diretório de entrada
dir_input = f'{dir}/output/glm_20s_goes'

# diretório de saída
dir_output = f'{dir}/output/glm_diario_goes'

# cria pasta de saída
os.makedirs(dir_output, exist_ok=True)

MessageError: Error: credential propagation was unsuccessful

# **2° Passo:** Declarando funções

In [ ]:
# Função que calcula o índice i e j da localização do relâmpago
def index(longitudes_matriz, latitudes_matriz, lon_raio, lat_raio):

    ''' Função para calcular o índice (i e j) do pixel de uma matriz que o relâmpago pertence

    Parâmetros:
               longitudes_matriz (array): array de uma dimensão das longitudes da matriz em graus
               latitudes_matriz (array): array de uma dimensão das latitudes da matriz em graus
               lon_raio (float): valor da longitude do relâmpago em graus
               lat_raio (float): valor da latitude do relâmpago em graus

    Retorna:
            indice_lat_raio (float): índice da latitude (ou seja, da linha) do pixel da matriz que o relâmpago pertence
            indice_lon_raio (float): índice da longitude (ou seja, da coluna) do pixel da matriz que o relâmpago pertence
    '''

    # calcula a diferença entre as lats/lons da matriz e a latitude/longitude do relâmpago
    distancia_lon = (longitudes_matriz - lon_raio)**2
    distancia_lat = (latitudes_matriz - lat_raio)**2

    # índice da longitude e latitude do relâmpago
    indice_lon_raio = np.nonzero(distancia_lon == np.min(distancia_lon))
    indice_lat_raio = np.nonzero(distancia_lat == np.min(distancia_lat))

    # retorna os valores dos índices calculados
    return indice_lat_raio, indice_lon_raio

# **Seleciona os flashes da área de interesse**


In [ ]:
%%time

# defina a DATA
data = '2020-06-30'

# lista os arquivos
files = sorted(glob.glob(f'{dir_input}/{data}/*/*.nc'))
#files=files[0:10]

# área desejada
lonmin, lonmax, latmin, latmax = -90, -30, -40, 10

# Lista para armazenar dados filtrados (mais eficiente que DataFrames)
filtered_data = []

# Otimização: pré-compilar condições de filtro
def in_region(lon, lat):
    return (lon > lonmin) & (lon < lonmax) & (lat > latmin) & (lat < latmax)

# Loop
for file in files:

    print(f'Processando: {file}')

    # Usando context manager para fechar arquivo automaticamente
    with xr.open_dataset(file) as glm_20s:

        # Acessando arrays diretamente (mais rápido que via DataFrame)
        lons_flash = glm_20s['flash_lon'].values
        lats_flash = glm_20s['flash_lat'].values
        times_flash = glm_20s['flash_time_offset_of_first_event'].values

        # Máscara booleana vetorizada (mais rápido que filtro no DataFrame)
        mask = in_region(lons_flash, lats_flash)

        # Apenas processa se houver dados na região
        if np.any(mask):

            # Cria DataFrame apenas com dados filtrados
            df_filtered = pd.DataFrame({'time': times_flash[mask],
                                        'lat': lats_flash[mask],
                                        'lon': lons_flash[mask]})

            filtered_data.append(df_filtered)

# concatenação final
df_flash_total = pd.concat(filtered_data, ignore_index=True) if filtered_data else pd.DataFrame(columns=['time', 'lat', 'lon'])

# salvar dataframe
df_flash_total.to_csv(f'{dir_output}/flash_glm_goes_{data}.csv', index=False)

In [ ]:
# mostra o dataframe final com a coluna time, lat e lon
df_flash_total

# **Acumula os flashes na grade e salva arquivo NETCDF**

In [ ]:
%%time
# espaçamento da grade
delta = 0.08   # grade com 8 km de resolução espacial

# montando a grade
lons = np.arange(lonmin, lonmax, delta)
lats = np.arange(latmin, latmax, delta)

# quantidade de pontos para longitude e latitude
nlon = len(lons)
nlat = len(lats)

# transforma de dataframe para numpy.array
flash_lon, flash_lat = df_flash_total['lon'].values, df_flash_total['lat'].values

# declara a matriz de relâmpagos
flashes = np.zeros((nlat, nlon))  # Exemplo: flash = np.zeros(qte_linha, qte_coluna)

# loop em cada longitude e latitude da lista
for lonraio, latraio in zip(flash_lon, flash_lat):

    # função que extrai a qual pixel (ou seja, determina as variáveis 'lin' e 'col') aquele relâmpago pertence
    lin, col = index(lons, lats, lonraio, latraio)

    # soma os relâmpagos por pixel
    flashes[lin,col]+=1

# gera netcdf
data_vars = {'flash':(('lat', 'lon'), flashes, {'units': 'ocorrências/64km²', 'long_name':'Flashes'})}
coords = {'lat': lats, 'lon': lons}
ds = xr.Dataset(data_vars=data_vars, coords=coords)
ds.to_netcdf(f'{dir_output}/flash_glm_goes_{data}.nc')

In [ ]:
# mostra o arquivo NETCDF que foi gerado
ds = xr.open_dataset(f'{dir_output}/flash_glm_goes_2020-06-30.nc')
ds

In [ ]:
# plota mapa simples deste arquivo NETCDF que foi gerado
ds['flash'].plot(vmin=1, vmax=100)